In [0]:
# -----------------------------------------------------------------------------------------------------------------------------------------------------
# SCRIPT: 3.1a_votacoes_ingest
# LOCAL: 1_bronze/src/3_correlacoes_frentes_votos/
# OBJETIVO: Ingestão de dados sobre votações/presenças parlamentares. Fonte p. 3_gold/../3_correlacoes_frentes_votos/3.1a_alinhemento_frente_partido
# ENTREGA: Verificar se deputados de uma mesma frente votam de forma mais alinhada do que seus colegas de partido.
# -----------------------------------------------------------------------------------------------------------------------------------------------------


import requests as req

url_base = "https://dadosabertos.camara.leg.br/api/v2/votacoes"
# Buscamos as últimas 10 votações para garantir que acharemos uma nominal
res = req.get(f"{url_base}?ordem=DESC&ordenarPor=dataHoraRegistro").json()['dados']

id_votacao_valida = None

print("Procurando votação com registros de votos...")

for v in res:
    id_v = v['id']
    # Checa se há votos para este ID
    check_votos = req.get(f"{url_base}/{id_v}/votos").json()['dados']
    
    if len(check_votos) > 0:
        id_votacao_valida = id_v
        dados_votos = check_votos
        # Busca orientações para esta mesma votação
        dados_orient = req.get(f"{url_base}/{id_v}/orientacoes").json()['dados']
        break

if id_votacao_valida:
    df_orient = spark.createDataFrame(dados_orient)
    df_votos = spark.createDataFrame(dados_votos)

    df_orient.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_orientacoes_partido")
    df_votos.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_votos_deputados")
    
    print(f"Sucesso! Encontrada a votação {id_votacao_valida} com {len(dados_votos)} votos.")
else:
    print("Nenhuma das últimas votações possui registros nominais. Tente aumentar o range ou verificar o período legislativo.")

In [0]:
display(spark.table("bronze_orientacoes_partido"))
display(spark.table("bronze_votos_deputados"))

In [0]:
import requests as req

# Acesso a API e recuperação dos dados
response = req.get("https://dadosabertos.camara.leg.br/api/v2/votacoes").json()
dados_lista = response['dados'] 

# Dataframe para salvar os dados
df_votacoes = spark.createDataFrame(dados_lista)

# Salvando os dados na tabela da camada bronze
df_votacoes.write.mode("overwrite").saveAsTable("bronze_votacoes")

In [0]:
display(spark.table("bronze_votacoes"))

In [0]:
# ----------------------------------------------------------------------------------------
# SCRIPT: 1_votacoes_ingest
# LOCAL: /1_bronze/src/
# OBJETIVO: Extração dos dados sobre votações
# ----------------------------------------------------------------------------------------

import requests as req

# 1. Buscar as últimas votações da legislatura atual
url_votacoes = "https://dadosabertos.camara.leg.br/api/v2/votacoes?ordem=DESC&ordenarPor=dataHoraRegistro"
res_votacoes = req.get(url_votacoes).json()['dados']

# Criar DataFrame de Votações
df_votacoes_bronze = spark.createDataFrame(res_votacoes)
df_votacoes_bronze.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_votacoes")

# 2. Para a análise ser rica, precisamos pegar os VOTOS individuais de uma votação importante
# Vamos pegar o ID da primeira votação da lista para testar a lógica
id_votacao = res_votacoes[0]['id']

url_votos = f"https://dadosabertos.camara.leg.br/api/v2/votacoes/{id_votacao}/votos"
res_votos = req.get(url_votos).json()['dados']

df_votos_bronze = spark.createDataFrame(res_votos)
df_votos_bronze.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_votos_detalhe")

print(f"Sucesso! Votação {id_votacao} carregada com seus votos individuais.")

In [0]:
display(spark.table("bronze_votacoes"))